In [1]:
import pyclifford as pc
import mcp_server as mcp
import numpy
from math import ceil

def sample_random_pauli_string(N):
    """Sample a random Pauli operator string with random phase for N qubits.
    Returns a string like '-iIXIIYZXIZ'.
    """
    paulis = ['I', 'X', 'Y', 'Z']
    phases = ['', 'i', '-', '-i']
    pauli_str = ''.join(numpy.random.choice(paulis) for _ in range(N))
    phase = numpy.random.choice(phases)
    return f'{phase}{pauli_str}'

def sample_question_and_answer(N):
    pauli_str1 = sample_random_pauli_string(N)
    pauli_str2 = sample_random_pauli_string(N)
    op1 = pc.pauli(pauli_str1)
    op2 = pc.pauli(pauli_str2)
    str1 = mcp.PauliTerm.from_obj(op1).text
    str2 = mcp.PauliTerm.from_obj(op2).text
    sol = mcp.PauliTerm.from_obj(op1 @ op2)
    return f'({str1}) ({str2})', sol

N_max = 10
iterations = 10


def generate_questions_and_answers(N_max, iterations):
    questions = []
    answers = []
    for i in range(iterations):
        q, a = sample_question_and_answer(N_max)
        questions.append(q)
        answers.append(a)

    questions_block = "\n".join(
        [f"({i+1}) Input: $ {q} $\n    Your Answer: " for i, q in enumerate(questions)]
    )

    pauli_instruction_template = f"""You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {{±1,±i}}. The Pauli operators obey the following multiplication rules:

    X_{{j}}^2 = Y_{{j}}^2 = Z_{{j}}^2 = I  
    X_{{j}}Y_{{j}} = iZ_{{j}}, Y_{{j}}Z_{{j}} = iX_{{j}}, Z_{{j}}X_{{j}} = iY_{{j}} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{{1}}Y_{{2}}Z_{{3}}), omitting identity operators. Do not include explanatory text or steps. Use LaTeX-style syntax, e.g., Z_{{1}}, X_{{2}} etc., and keep one space between operators.

    Examples:

    (1) Input: $(+iZ_{{1}})(−iX_{{1}})$  
        Answer: +iY_{{1}}

    (2) Input: $(+iZ_{{1}}Y_{{2}})(Z_{{1}}X_{{2}})$  
        Answer: Z_{{2}}

    Questions (Compute and give your answer in the same format as above):

    {questions_block}

    ---

    After answering all questions, output your answers as a Python list of strings for easy copying and checking.

    Please follow this exact format (note: this is an example, not related to the above questions):

    ```python
    LLM_answers = [
        '-i Y_{0}',
        '- X_{0} Z_{1}',
        'X_{0} Y_{1} Y_{2}',
        '- Y_{1} Z_{2} Y_{3}',
        '- Y_{0} X_{1} Y_{4}',
        '-i Z_{2} Z_{3} Z_{4} Z_{5} X_{6}',
        '+i Y_{0} Z_{1} Y_{2} X_{3} X_{6}',
        '+i Z_{0} Z_{1} Y_{2} Y_{3} Y_{4} Z_{6} Y_{8}',
        'Z_{1} X_{5} Z_{6} X_{7} Y_{8}',
        ...
    ]"""

    return pauli_instruction_template, answers

## Without MCP tools:

### ChatGPT

#### o3

In [2]:
N_max = 1
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [3]:
answers

[PauliTerm(coefficient=(-1+0j), pauli_string={}, text='- I'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y'}, text='+i Y_{0}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Z'}, text='Z_{0}'),
 PauliTerm(coefficient=(1+0j), pauli_string={}, text='I'),
 PauliTerm(coefficient=(-1+0j), pauli_string={}, text='- I'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z'}, text='+i Z_{0}'),
 PauliTerm(coefficient=1j, pauli_string={}, text='+i I'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X'}, text='+i X_{0}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Z'}, text='Z_{0}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X'}, text='-i X_{0}')]

In [4]:
LLM_answers = [
    '-1',
    '+i Y_0',
    'Z_0',
    '+1',
    '-1',
    '+i Z_0',
    '+i',
    '+i X_0',
    'Z_0',
    '-i X_0',
]

In [5]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 1, question_volume  = 10, accuracy = 1.0


In [6]:
accuracy

[True, True, True, True, True, True, True, True, True, True]

________________________________________

In [7]:
N_max = 2
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [8]:
answers

[PauliTerm(coefficient=(-1+0j), pauli_string={0: 'X', 1: 'Z'}, text='- X_{0} Z_{1}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 1: 'Y'}, text='+i X_{0} Y_{1}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={}, text='-i I'),
 PauliTerm(coefficient=1j, pauli_string={1: 'Z'}, text='+i Z_{1}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 1: 'X'}, text='-i X_{0} X_{1}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={1: 'X'}, text='-i X_{1}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 1: 'Y'}, text='-i Y_{0} Y_{1}'),
 PauliTerm(coefficient=1j, pauli_string={1: 'Y'}, text='+i Y_{1}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 1: 'Y'}, text='-i X_{0} Y_{1}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Z', 1: 'Y'}, text='-i Z_{0} Y_{1}')]

In [9]:
LLM_answers = [
    '- X_0 Z_1',
    '+i X_0 Y_1',
    '-i',
    '+i Z_1',
    '-i X_0 X_1',
    '-i X_1',
    '-i Y_0 Y_1',
    '+i Y_1',
    '-i X_0 Y_1',
    '-i Z_0 Y_1',
]

In [10]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 2, question_volume  = 10, accuracy = 1.0


In [11]:
accuracy

[True, True, True, True, True, True, True, True, True, True]

________________________________________

In [12]:
N_max = 3
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [13]:
answers

[PauliTerm(coefficient=(-0-1j), pauli_string={1: 'X', 2: 'Y'}, text='-i X_{1} Y_{2}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'Y'}, text='Y_{1}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X'}, text='-i X_{0}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={1: 'Y', 2: 'Z'}, text='- Y_{1} Z_{2}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Y', 1: 'Z', 2: 'Z'}, text='- Y_{0} Z_{1} Z_{2}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y', 1: 'Y', 2: 'Z'}, text='+i Y_{0} Y_{1} Z_{2}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={2: 'Y'}, text='-i Y_{2}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={2: 'X'}, text='- X_{2}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Y', 1: 'Y'}, text='Y_{0} Y_{1}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 1: 'Y'}, text='+i X_{0} Y_{1}')]

In [14]:
LLM_answers = [
    '-i X_1 Y_2',
    'Y_1',
    '-i X_0',
    '- Y_1 Z_2',
    '- Y_0 Z_1 Z_2',
    '+i Y_0 Y_1 Z_2',
    '-i Y_2',
    '- X_2',
    'Y_0 Y_1',
    '+i X_0 Y_1',
]

In [15]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 3, question_volume  = 10, accuracy = 1.0


In [16]:
accuracy

[True, True, True, True, True, True, True, True, True, True]

________________________________________

In [17]:
N_max = 4
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [18]:
answers

[PauliTerm(coefficient=(-1+0j), pauli_string={1: 'Z', 2: 'Z', 3: 'X'}, text='- Z_{1} Z_{2} X_{3}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'X', 1: 'Y', 2: 'Y', 3: 'Z'}, text='X_{0} Y_{1} Y_{2} Z_{3}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Z', 1: 'Y', 2: 'X', 3: 'Y'}, text='-i Z_{0} Y_{1} X_{2} Y_{3}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z', 1: 'Z', 2: 'X', 3: 'X'}, text='+i Z_{0} Z_{1} X_{2} X_{3}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 2: 'X', 3: 'Z'}, text='+i X_{0} X_{2} Z_{3}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 1: 'X', 3: 'Z'}, text='+i X_{0} X_{1} Z_{3}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={1: 'Z', 2: 'Y', 3: 'Z'}, text='-i Z_{1} Y_{2} Z_{3}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 2: 'Y'}, text='-i X_{0} Y_{2}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Z', 1: 'Z', 2: 'Z', 3: 'X'}, text='- Z_{0} Z_{1} Z_{2} X_{3}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Z', 2: 'Y', 3: 'Y'}, 

In [19]:
LLM_answers = [
    '- Z_1 Z_2 X_3',
    'X_0 Y_1 Y_2 Z_3',
    '-i Z_0 Y_1 X_2 Y_3',
    '+i Z_0 Z_1 X_2 X_3',
    '+i X_0 X_2 Z_3',
    '+i X_0 X_1 Z_3',
    '-i Z_1 Y_2 Z_3',
    '-i X_0 Y_2',
    '- Z_0 Z_1 Z_2 X_3',
    'Z_0 Y_2 Y_3',
]

In [20]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 4, question_volume  = 10, accuracy = 1.0


In [21]:
accuracy

[True, True, True, True, True, True, True, True, True, True]

________________________________________

In [22]:
N_max = 5
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [23]:
answers

[PauliTerm(coefficient=1j, pauli_string={0: 'Y', 1: 'Y', 2: 'Z', 3: 'Y', 4: 'Z'}, text='+i Y_{0} Y_{1} Z_{2} Y_{3} Z_{4}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'Y', 2: 'Z', 3: 'Z', 4: 'Z'}, text='Y_{1} Z_{2} Z_{3} Z_{4}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Z', 1: 'Z', 2: 'X', 3: 'Y', 4: 'Z'}, text='Z_{0} Z_{1} X_{2} Y_{3} Z_{4}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 1: 'Y', 3: 'Y', 4: 'X'}, text='-i X_{0} Y_{1} Y_{3} X_{4}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 1: 'Z', 3: 'X', 4: 'X'}, text='+i X_{0} Z_{1} X_{3} X_{4}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 1: 'Z', 2: 'Z', 3: 'Z', 4: 'Z'}, text='-i Y_{0} Z_{1} Z_{2} Z_{3} Z_{4}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Z', 1: 'X', 2: 'Z', 4: 'Z'}, text='Z_{0} X_{1} Z_{2} Z_{4}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Z', 1: 'Y', 2: 'Z', 4: 'Z'}, text='- Z_{0} Y_{1} Z_{2} Z_{4}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Z', 2: 'X',

In [24]:
LLM_answers = [
    '+i Y_0 Y_1 Z_2 Y_3 Z_4',
    'Y_1 Z_2 Z_3 Z_4',
    'Z_0 Z_1 X_2 Y_3 Z_4',
    '-i X_0 Y_1 Y_3 X_4',
    '+i X_0 Z_1 X_3 X_4',
    '-i Y_0 Z_1 Z_2 Z_3 Z_4',
    'Z_0 X_1 Z_2 Z_4',
    '- Z_0 Y_1 Z_2 Z_4',
    '-i Z_0 X_2 Y_3',
    '-i Y_0 Y_1 X_3 Y_4',
]

In [25]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 5, question_volume  = 10, accuracy = 1.0


In [26]:
accuracy

[True, True, True, True, True, True, True, True, True, True]

________________________________________

In [27]:
N_max = 6
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [28]:
answers

[PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 1: 'Y', 2: 'X', 3: 'Y', 5: 'X'}, text='-i Y_{0} Y_{1} X_{2} Y_{3} X_{5}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={2: 'Z', 4: 'Y', 5: 'Y'}, text='-i Z_{2} Y_{4} Y_{5}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y', 1: 'Z', 2: 'Y', 3: 'X', 5: 'Y'}, text='+i Y_{0} Z_{1} Y_{2} X_{3} Y_{5}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Y', 1: 'Z', 2: 'Z', 3: 'X', 5: 'Z'}, text='Y_{0} Z_{1} Z_{2} X_{3} Z_{5}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Z', 1: 'X', 2: 'X', 3: 'Z', 4: 'Y', 5: 'Z'}, text='Z_{0} X_{1} X_{2} Z_{3} Y_{4} Z_{5}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'X', 1: 'X', 3: 'Y', 5: 'Y'}, text='- X_{0} X_{1} Y_{3} Y_{5}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Z', 1: 'Y', 2: 'X', 3: 'X', 4: 'Y'}, text='+i Z_{0} Y_{1} X_{2} X_{3} Y_{4}'),
 PauliTerm(coefficient=1j, pauli_string={2: 'Y', 4: 'Y', 5: 'X'}, text='+i Y_{2} Y_{4} X_{5}'),
 PauliTerm(coefficient=1j, pauli_string={2: 'X',

In [29]:
LLM_answers = [
    '-i Y_0 Y_1 X_2 Y_3 X_5',
    '-i Z_2 Y_4 Y_5',
    '+i Y_0 Z_1 Y_2 X_3 Y_5',
    'Y_0 Z_1 Z_2 X_3 Z_5',
    'Z_0 X_1 X_2 Z_3 Y_4 Z_5',
    '- X_0 X_1 Y_3 Y_5',
    '+i Z_0 Y_1 X_2 X_3 Y_4',
    '+i Y_2 Y_4 X_5',
    '+i X_2 Z_3 Z_5',
    '-i Y_0 Y_1 X_4',
]

In [30]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 6, question_volume  = 10, accuracy = 1.0


In [31]:
accuracy

[True, True, True, True, True, True, True, True, True, True]

________________________________________

In [32]:
N_max = 7
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [33]:
answers

[PauliTerm(coefficient=(-1+0j), pauli_string={0: 'Z', 1: 'Y', 2: 'Y', 3: 'Y', 5: 'Z', 6: 'X'}, text='- Z_{0} Y_{1} Y_{2} Y_{3} Z_{5} X_{6}'),
 PauliTerm(coefficient=1j, pauli_string={1: 'Z', 2: 'X', 4: 'X', 5: 'Z'}, text='+i Z_{1} X_{2} X_{4} Z_{5}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={2: 'X', 3: 'X', 4: 'Z', 5: 'Z', 6: 'X'}, text='-i X_{2} X_{3} Z_{4} Z_{5} X_{6}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 1: 'Z', 2: 'X', 4: 'Z', 5: 'X'}, text='-i Y_{0} Z_{1} X_{2} Z_{4} X_{5}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Z', 1: 'X', 2: 'X', 3: 'Z', 4: 'Z', 5: 'X', 6: 'Z'}, text='Z_{0} X_{1} X_{2} Z_{3} Z_{4} X_{5} Z_{6}'),
 PauliTerm(coefficient=1j, pauli_string={1: 'X', 2: 'Y', 3: 'Y', 4: 'Z', 5: 'Y', 6: 'Z'}, text='+i X_{1} Y_{2} Y_{3} Z_{4} Y_{5} Z_{6}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 1: 'Y', 2: 'X', 3: 'Z', 4: 'Y', 6: 'Z'}, text='-i Y_{0} Y_{1} X_{2} Z_{3} Y_{4} Z_{6}'),
 PauliTerm(coefficient=1j, pauli_string={1: 'Y', 2: 'Y',

In [34]:
LLM_answers = [
    '- Z_0 Y_1 Y_2 Y_3 Z_5 X_6',
    '+i Z_1 X_2 X_4 Z_5',
    '-i X_2 X_3 Z_4 Z_5 X_6',
    '-i Y_0 Z_1 X_2 Z_4 X_5',
    'Z_0 X_1 X_2 Z_3 Z_4 X_5 Z_6',
    '+i X_1 Y_2 Y_3 Z_4 Y_5 Z_6',
    '-i Y_0 Y_1 X_2 Z_3 Y_4 Z_6',
    '+i Y_1 Y_2 X_4 X_5 Z_6',
    '-i X_0 Z_1 Y_2 X_4 Y_5',
    'Y_0 Z_1 Y_2 Z_4 X_5 X_6',
]

In [35]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 7, question_volume  = 10, accuracy = 1.0


________________________________________

In [36]:
N_max = 8
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [37]:
answers

[PauliTerm(coefficient=1j, pauli_string={0: 'Y', 1: 'Z', 3: 'X', 5: 'Y', 6: 'Z', 7: 'X'}, text='+i Y_{0} Z_{1} X_{3} Y_{5} Z_{6} X_{7}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={2: 'X', 3: 'Z', 4: 'Y', 5: 'X', 7: 'X'}, text='- X_{2} Z_{3} Y_{4} X_{5} X_{7}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 1: 'Z', 5: 'Y', 6: 'X'}, text='-i Y_{0} Z_{1} Y_{5} X_{6}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 1: 'X', 2: 'Y', 3: 'Y', 4: 'Y', 6: 'Y', 7: 'Z'}, text='-i Y_{0} X_{1} Y_{2} Y_{3} Y_{4} Y_{6} Z_{7}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y', 1: 'X', 2: 'Z', 3: 'Y', 4: 'Z', 5: 'Z'}, text='+i Y_{0} X_{1} Z_{2} Y_{3} Z_{4} Z_{5}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'X', 1: 'Z', 3: 'X', 5: 'Z', 6: 'X', 7: 'X'}, text='+i X_{0} Z_{1} X_{3} Z_{5} X_{6} X_{7}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y', 2: 'Z', 3: 'Z', 5: 'Y', 6: 'X', 7: 'Z'}, text='-i Y_{0} Z_{2} Z_{3} Y_{5} X_{6} Z_{7}'),
 PauliTerm(coefficient=(-0-1j), pauli_strin

In [38]:
LLM_answers = [
    '+i Y_0 Z_1 X_3 Y_5 Z_6 X_7',
    '- X_2 Z_3 Y_4 X_5 X_7',
    '-i Y_0 Z_1 Y_5 X_6',
    '-i Y_0 X_1 Y_2 Y_3 Y_4 Y_6 Z_7',
    '+i Y_0 X_1 Z_2 Y_3 Z_4 Z_5',
    '+i X_0 Z_1 X_3 Z_5 X_6 X_7',
    '-i Y_0 Z_2 Z_3 Y_5 X_6 Z_7',
    '-i Y_0 Z_1 Y_2 Z_3 X_4 X_5 Y_6 Z_7',
    '-i X_0 Y_1 Y_3 X_4 Z_5 Z_7',
    '- X_0 Z_1 Y_2 Y_3 Y_5 Z_7',
]

In [39]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 8, question_volume  = 10, accuracy = 1.0


________________________________________

In [40]:
N_max = 9
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [41]:
answers

[PauliTerm(coefficient=(1+0j), pauli_string={0: 'Y', 2: 'X', 3: 'Z', 4: 'X', 5: 'Y', 6: 'Z', 8: 'X'}, text='Y_{0} X_{2} Z_{3} X_{4} Y_{5} Z_{6} X_{8}'),
 PauliTerm(coefficient=1j, pauli_string={2: 'Z', 3: 'Z', 4: 'Z', 5: 'X', 6: 'Y', 7: 'Y', 8: 'Z'}, text='+i Z_{2} Z_{3} Z_{4} X_{5} Y_{6} Y_{7} Z_{8}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'Y', 2: 'X', 3: 'Y', 4: 'X', 6: 'Z', 7: 'Z', 8: 'X'}, text='Y_{1} X_{2} Y_{3} X_{4} Z_{6} Z_{7} X_{8}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0: 'X', 1: 'X', 2: 'X', 3: 'Z', 4: 'Z', 5: 'Y', 7: 'X'}, text='- X_{0} X_{1} X_{2} Z_{3} Z_{4} Y_{5} X_{7}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Y', 1: 'Y', 3: 'Y', 5: 'Z', 6: 'X', 7: 'Y', 8: 'Z'}, text='Y_{0} Y_{1} Y_{3} Z_{5} X_{6} Y_{7} Z_{8}'),
 PauliTerm(coefficient=1j, pauli_string={0: 'Y', 1: 'Z', 2: 'X', 3: 'Y', 4: 'Z', 5: 'Z', 6: 'X', 7: 'Y', 8: 'Z'}, text='+i Y_{0} Z_{1} X_{2} Y_{3} Z_{4} Z_{5} X_{6} Y_{7} Z_{8}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'Y',

In [42]:
LLM_answers = [
    'Y_0 X_2 Z_3 X_4 Y_5 Z_6 X_8',
    '+i Z_2 Z_3 Z_4 X_5 Y_6 Y_7 Z_8',
    'Y_1 X_2 Y_3 X_4 Z_6 Z_7 X_8',
    '- X_0 X_1 X_2 Z_3 Z_4 Y_5 X_7',
    'Y_0 Y_1 Y_3 Z_5 X_6 Y_7 Z_8',
    '+i Y_0 Z_1 X_2 Y_3 Z_4 Z_5 X_6 Y_7 Z_8',
    '-i Y_0 Y_1 Z_2 Z_3 X_5 Y_6 Y_7 X_8',
    '-i Y_0 Y_2 Y_4 Y_5 Y_7',
    '+i Z_1 Y_3 X_4 X_8',
    'Y_1 X_3 X_4 Z_5 Z_6 Z_7',
]

In [43]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 9, question_volume  = 10, accuracy = 1.0


________________________________________

In [44]:
N_max = 10
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [45]:
answers

[PauliTerm(coefficient=(-1+0j), pauli_string={1: 'X', 2: 'Y', 3: 'X', 4: 'X', 5: 'Y', 8: 'X', 9: 'Y'}, text='- X_{1} Y_{2} X_{3} X_{4} Y_{5} X_{8} Y_{9}'),
 PauliTerm(coefficient=1j, pauli_string={1: 'Z', 3: 'Z', 6: 'Z', 7: 'X', 8: 'X'}, text='+i Z_{1} Z_{3} Z_{6} X_{7} X_{8}'),
 PauliTerm(coefficient=(1+0j), pauli_string={1: 'Z', 2: 'X', 4: 'X', 5: 'Y', 6: 'Y', 8: 'Z', 9: 'Z'}, text='Z_{1} X_{2} X_{4} Y_{5} Y_{6} Z_{8} Z_{9}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'X', 1: 'X', 2: 'X', 3: 'Z', 4: 'Y', 5: 'X', 6: 'Y', 7: 'Y', 9: 'Z'}, text='X_{0} X_{1} X_{2} Z_{3} Y_{4} X_{5} Y_{6} Y_{7} Z_{9}'),
 PauliTerm(coefficient=(1+0j), pauli_string={0: 'Y', 1: 'Z', 3: 'Z', 4: 'Y', 5: 'Z', 6: 'X', 7: 'X', 8: 'Y', 9: 'X'}, text='Y_{0} Z_{1} Z_{3} Y_{4} Z_{5} X_{6} X_{7} Y_{8} X_{9}'),
 PauliTerm(coefficient=(-0-1j), pauli_string={0: 'X', 1: 'X', 2: 'Z', 4: 'Y', 6: 'X', 7: 'X', 8: 'Z'}, text='-i X_{0} X_{1} Z_{2} Y_{4} X_{6} X_{7} Z_{8}'),
 PauliTerm(coefficient=(-1+0j), pauli_string={0:

In [46]:
LLM_answers = [
    '- X_1 Y_2 X_3 X_4 Y_5 X_8 Y_9',
    '+i Z_1 Z_3 Z_6 X_7 X_8',
    'Z_1 X_2 X_4 Y_5 Y_6 Z_8 Z_9',
    'X_0 X_1 X_2 Z_3 Y_4 X_5 Y_6 Y_7 Z_9',
    'Y_0 Z_1 Z_3 Y_4 Z_5 X_6 X_7 Y_8 X_9',
    '-i X_0 X_1 Z_2 Y_4 X_6 X_7 Z_8',
    '- X_0 X_1 X_2 X_3 Z_4 X_5 X_6 X_7 X_9',
    '+i X_0 X_1 Y_3 X_4 Z_6 Z_7 Y_8',
    '+i Z_0 X_1 Y_2 X_4 Z_5 Y_6 Y_8 Y_9',
    '- X_2 Z_3 X_4 Y_5 Y_6 X_7 Y_8 Y_9',
]

In [47]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 10, question_volume  = 10, accuracy = 1.0


________________________________________

In [52]:
N_max = 20
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [53]:
LLM_answers = [
    'X_0 Y_1 Z_2 Z_3 X_4 Y_5 Y_6 Y_7 X_9 Z_11 X_14 Z_15 Z_16 Y_17 Y_18 Z_19',
    'Y_0 Z_1 Z_2 Z_4 Z_5 Y_6 Y_7 X_9 X_10 X_11 Y_12 Y_13 Y_14 Z_15 X_17',
    '+i X_0 X_1 X_3 X_4 Z_7 X_8 Y_9 Z_11 X_12 Y_14 Z_17 X_18 Y_19',
    '- X_1 Y_2 Z_3 Z_4 X_5 X_6 X_7 Z_8 X_9 Z_10 X_12 X_13 Z_14 Z_16 Y_17 Y_18 X_19',
    'Y_0 Y_1 Y_2 Z_3 Z_6 Z_7 X_8 Y_9 X_10 Z_12 X_13 Z_14 Y_15 Z_16 Y_17 X_18 X_19',
    '- Y_1 Z_3 Z_4 Z_5 Y_6 Z_7 Y_8 Y_9 Y_10 Y_11 Z_12 Y_13 Y_14 Z_15 X_17 X_19',
    '-i Y_2 Y_3 X_4 Y_5 Y_6 Z_8 Y_9 Z_10 Y_11 Z_12 Z_13 Z_14 Y_15 Y_16 Y_17 X_18 X_19',
    '-i Y_0 Z_3 Y_4 Y_5 Y_8 Z_10 Y_12 Z_13 Y_14 Y_15 Z_16 Y_18 Y_19',
    '+i Y_0 Y_1 Y_2 X_3 X_5 X_6 X_7 X_8 Y_9 Z_10 Z_11 Y_12 Y_13 X_15 Y_16 Y_17 Z_18 X_19',
    '- Y_0 Z_2 X_4 Z_5 Y_6 X_8 Y_9 Y_10 X_12 Z_13 Y_16 Y_17 Y_18 Z_19',
]

In [54]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 20, question_volume  = 10, accuracy = 1.0


________________________________________

In [60]:
N_max = 30
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [61]:
LLM_answers = [
    '- X_{0} Y_{4} X_{5} Z_{6} X_{7} X_{9} X_{10} Z_{11} Z_{12} Z_{14} Z_{15} X_{16} Z_{17} Z_{19} Z_{23} Z_{24} Y_{25} Z_{26} X_{27} Y_{29}',
    'Z_{0} X_{2} X_{3} Y_{4} Z_{5} X_{7} Z_{8} Y_{9} X_{10} Y_{11} Z_{12} Z_{13} X_{14} Y_{15} Z_{16} Z_{17} Z_{18} Z_{20} X_{21} X_{24} X_{25} Z_{27} X_{28} Y_{29}',
    '-i X_{0} X_{4} Y_{5} Y_{6} Y_{7} Z_{8} X_{10} Z_{12} Z_{13} Y_{15} Y_{16} Y_{18} X_{19} Y_{20} X_{21} Z_{22} X_{23} X_{24} Z_{25} Y_{26} Z_{27} Z_{28}',
    '-i X_{0} X_{1} X_{3} Z_{4} Y_{5} Y_{6} Z_{7} X_{8} Y_{9} X_{11} Z_{13} X_{14} X_{15} Y_{17} X_{18} X_{19} X_{20} Z_{21} Z_{22} Z_{23} X_{25} X_{26} Z_{29}',
    '- Y_{0} X_{1} Z_{2} X_{5} Y_{6} X_{8} X_{11} Y_{12} Z_{13} Y_{14} Y_{15} X_{16} X_{17} Z_{19} Z_{21} X_{22} Y_{23} Z_{24} Y_{25} Y_{27} X_{28} X_{29}',
    '-i Y_{0} X_{1} X_{2} X_{3} Z_{4} Z_{5} Y_{6} Z_{7} Y_{8} Y_{9} X_{10} Y_{11} Z_{12} X_{13} X_{14} X_{17} Y_{18} Y_{19} X_{20} X_{22} Y_{23} Z_{24} Y_{25} Z_{26} Y_{27} X_{28} X_{29}',
    '- X_{1} Z_{5} Z_{7} X_{8} X_{9} X_{10} X_{11} Z_{12} Y_{13} Y_{14} Y_{15} Z_{18} X_{19} X_{20} Y_{21} Z_{22} Z_{23} Z_{24} X_{26} X_{27} X_{28} Z_{29}',
    'Z_{3} Z_{4} Z_{5} X_{6} X_{7} Z_{9} Y_{10} X_{11} Y_{13} X_{14} Y_{17} Z_{18} Y_{20} X_{22} Y_{23} Y_{24} Z_{26} X_{28} Z_{29}',
    '-i X_{0} Y_{3} Z_{4} Y_{5} X_{6} Z_{7} X_{8} X_{11} X_{12} Y_{14} Y_{16} Y_{17} X_{19} Z_{20} Z_{21} Z_{23} Z_{24} X_{26}',
    '-i Y_{0} X_{1} Y_{2} X_{3} Y_{4} X_{5} X_{8} Y_{9} X_{11} Y_{12} Z_{13} Z_{14} Y_{15} Y_{16} Y_{19} X_{20} X_{21} Z_{22} Z_{25} X_{26} Y_{27} Z_{28}',
]

In [62]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 30, question_volume  = 10, accuracy = 1.0


________________________________________

In [63]:
N_max = 35
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [64]:
LLM_answers = [
    'Z_{0} Y_{2} Z_{3} Y_{4} Y_{5} Z_{7} X_{8} Z_{10} X_{11} Z_{12} X_{14} X_{17} Z_{18} X_{19} Y_{20} Z_{21} Y_{22} Z_{24} Y_{25} X_{27} Z_{29} Y_{31} X_{32} Z_{33} Z_{34}',
    'Z_{0} X_{1} Y_{3} Z_{5} Y_{7} X_{10} Y_{11} X_{12} Y_{13} X_{14} Z_{15} Y_{16} X_{18} X_{19} Y_{20} Z_{21} Z_{22} X_{23} X_{24} X_{25} Z_{27} Y_{28} X_{29} Y_{30} Y_{31} X_{32} Z_{33} Z_{34}',
    'X_{1} Y_{2} Z_{4} Z_{5} Z_{6} X_{7} Z_{9} X_{10} Z_{11} Y_{12} X_{13} X_{14} Z_{15} Z_{16} Z_{17} Y_{18} X_{19} Y_{20} Z_{21} X_{22} Y_{23} X_{24} X_{25} Z_{26} Y_{27} Y_{28} Z_{29} Z_{30} Z_{31} Z_{32} X_{33} Z_{34}',
    'X_{0} X_{1} X_{3} Y_{5} Y_{6} Z_{7} Z_{8} X_{9} Z_{10} Y_{11} Z_{12} Z_{13} X_{14} Z_{15} Z_{16} Y_{18} X_{19} Y_{20} Z_{21} Z_{22} X_{23} Z_{24} X_{25} X_{26} Y_{28} Z_{29} Z_{30} Z_{31} Y_{32} Z_{34}',
    '+i Z_{0} Y_{1} Y_{2} Z_{4} X_{6} Y_{7} Z_{9} X_{10} X_{11} Z_{13} Y_{14} Z_{15} X_{16} Y_{17} X_{18} X_{19} Z_{20} X_{22} Z_{23} Y_{24} Z_{25} Y_{26} Y_{29} Z_{30} Y_{32} Z_{33} Y_{34}',
    '- Z_{2} Z_{4} Y_{5} Y_{6} Z_{7} Y_{8} Z_{10} X_{11} X_{12} Y_{13} Z_{14} Z_{15} Z_{16} Y_{17} Z_{18} X_{19} Y_{23} X_{24} Z_{25} Z_{26} X_{27} Y_{29} X_{30} Y_{31} Y_{32}',
    '- X_{0} X_{1} X_{3} Y_{5} X_{7} Y_{8} Z_{12} Y_{13} Z_{14} Z_{15} X_{16} X_{17} Y_{18} Y_{21} Z_{22} X_{23} Y_{25} X_{26} Y_{27} Z_{28} X_{29} X_{30} X_{31} X_{32} X_{33}',
    '-i Y_{1} Z_{2} Y_{4} Y_{5} X_{6} Z_{7} X_{8} Z_{9} Y_{10} Y_{11} X_{13} X_{14} Z_{15} Y_{17} X_{18} Z_{19} X_{20} X_{21} Z_{22} X_{23} Z_{26} Z_{29} Y_{30} Y_{31} Z_{32} X_{33} Y_{34}',
    '+i X_{1} Z_{2} X_{4} Z_{5} X_{6} Y_{7} Z_{8} Z_{9} X_{10} Y_{11} Z_{12} X_{13} Z_{14} X_{15} Z_{16} Z_{17} X_{18} Z_{19} X_{20} Z_{22} Z_{24} X_{25} Z_{26} Y_{27} X_{29} Y_{30} Y_{31} Z_{32} Y_{33} Z_{34}',
    'X_{4} X_{5} Y_{6} X_{7} X_{8} Z_{9} X_{10} Z_{11} Z_{13} Z_{15} Y_{16} Y_{17} X_{18} Y_{19} X_{21} Y_{22} X_{23} X_{26} X_{27} Y_{28} Y_{29} Y_{30}'
]

In [65]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 35, question_volume  = 10, accuracy = 0.1


________________________________________

In [55]:
N_max = 40
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [58]:
LLM_answers = [
    'X_{3} Y_{4} X_{7} Y_{8} Z_{9} X_{10} Y_{11} Z_{12} Y_{13} Z_{14} Y_{15} Y_{16} Y_{17} Y_{18} X_{20} Z_{21} X_{22} Z_{23} Z_{24} X_{25} X_{26} Y_{28} Z_{30} Z_{31} X_{32} Z_{34} Z_{35} Z_{36} Z_{38} Y_{39}',
    '-i Y_{0} Z_{1} X_{2} X_{3} X_{5} Z_{6} Z_{7} X_{8} X_{9} Y_{10} Z_{11} Z_{12} Y_{13} Z_{14} Y_{15} Y_{16} Z_{17} X_{18} X_{19} Z_{20} Z_{21} X_{23} Y_{24} Z_{25} Y_{27} Y_{28} X_{29} Y_{30} Y_{31} X_{32} Y_{33} X_{34} X_{35} Y_{36} Z_{39}',
    'X_{0} Y_{1} Z_{4} Y_{5} Z_{6} Y_{7} Z_{8} X_{9} Z_{10} X_{12} Z_{13} Z_{14} X_{15} Z_{17} Z_{18} Z_{19} X_{20} Z_{21} Y_{22} X_{23} X_{24} Z_{25} Z_{28} Z_{29} Y_{30} Z_{31} Y_{32} X_{34} Y_{35} Y_{37} Y_{39}',
    'Y_{1} Z_{3} Z_{4} X_{5} X_{6} Z_{7} Y_{10} Y_{11} Z_{12} Z_{13} X_{15} Y_{16} Z_{17} Z_{18} X_{19} X_{20} X_{21} Z_{22} Z_{23} X_{24} X_{26} Z_{27} Y_{29} Z_{30} Z_{31} X_{32} Z_{33} Y_{34} Z_{36} Y_{39}',
    'Z_{0} X_{1} X_{3} Y_{4} Y_{5} Z_{6} Z_{7} Z_{8} X_{9} Z_{10} Y_{11} Z_{12} X_{13} Y_{15} Z_{16} Y_{17} Z_{20} Y_{23} Z_{24} X_{25} Z_{26} Y_{28} X_{29} X_{30} Y_{31} Y_{32} Y_{33} Y_{34} X_{35} Y_{37} X_{38} X_{39}',
    'X_{1} Y_{2} Y_{3} X_{5} X_{6} Y_{8} X_{11} Z_{12} Y_{14} Y_{16} Z_{17} X_{18} X_{19} Z_{20} Z_{21} Z_{23} Y_{25} X_{28} X_{29} Y_{31} X_{32} Y_{33} X_{34} Z_{35} Z_{37}',
    '+i Z_{0} Y_{3} X_{4} X_{5} X_{6} X_{7} Y_{8} X_{9} Y_{10} X_{11} X_{12} Z_{13} Y_{14} X_{16} X_{18} Y_{19} X_{20} X_{21} Z_{22} Z_{23} X_{24} X_{26} Z_{27} Y_{28} Z_{31} Y_{32} Y_{33} Z_{34} X_{35} Z_{36} X_{37} Y_{38} X_{39}',
    '-i Z_{0} Z_{2} Y_{4} X_{5} Z_{6} X_{7} X_{8} Z_{9} Z_{10} Z_{11} Z_{12} X_{14} Z_{15} Y_{19} X_{20} X_{21} Z_{23} X_{25} X_{27} Z_{29} X_{31} Y_{32} Z_{33} X_{34} Y_{36} Z_{37} Z_{38} Z_{39}',
    'X_{1} X_{2} X_{3} X_{4} Z_{5} X_{6} Z_{7} Y_{8} Y_{12} Z_{13} X_{14} Z_{15} Z_{16} X_{19} Y_{22} X_{23} X_{24} Y_{25} Y_{26} Y_{28} X_{29} Z_{30} X_{31} X_{32} Y_{34} X_{35} Z_{36} X_{38} X_{39}',
    '+i X_{0} Z_{1} Y_{2} X_{3} Z_{5} Z_{6} X_{7} X_{8} X_{10} Z_{11} X_{13} Y_{14} X_{15} Z_{16} Z_{17} X_{18} X_{19} X_{21} Y_{22} Y_{23} Y_{24} Y_{25} Y_{26} Y_{28} Y_{29} Y_{30} Y_{31} X_{33} Y_{35} Z_{36} X_{37} Y_{38} Z_{39}',
]

In [59]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 40, question_volume  = 10, accuracy = 0.1


________________________________________

In [48]:
N_max = 100
iterations = 10

questions, answers = generate_questions_and_answers(N_max, iterations)
print(questions)

You are given two strings of Pauli operators acting on qubits indexed by subscripts. Each operator string may also include a global phase factor from the set {±1,±i}. The Pauli operators obey the following multiplication rules:

    X_{j}^2 = Y_{j}^2 = Z_{j}^2 = I  
    X_{j}Y_{j} = iZ_{j}, Y_{j}Z_{j} = iX_{j}, Z_{j}X_{j} = iY_{j} (and cyclic permutations)  
    Operators on the same qubit anticommute.  
    Operators on different qubits commute.  

    Your task is to compute the product of the two operator strings by applying the Pauli algebra. The result should be a single simplified operator string, combining operators acting on the same qubit where applicable, and simplifying all coefficients to a single phase factor (±1, ±i). You must preserve the order of qubit indices in ascending order in your final expression.

    Format your answer as a single operator string, with phase factor in front (e.g., -i X_{1}Y_{2}Z_{3}), omitting identity operators. Do not include explanatory text

In [49]:
LLM_answers = [
    '+i X_0 Z_1 Y_5 Z_8 X_11 Y_17 Z_18 X_23 X_28 Y_31 Z_33 X_38 Y_40 Z_44 Y_47 X_49 Z_53 Y_56 X_61 Z_66 Y_69 X_70 Y_73 Z_75 Y_76 Z_81 X_83 Z_85 X_88 Z_91 Y_94',
    '+ Y_0 X_1 Z_2 X_4 Z_6 Y_7 X_8 Z_9 Y_11 X_14 Z_17 Y_18 X_22 Z_24 X_26 Y_29 Z_30 X_33 Y_35 X_38 Z_40 Y_43 X_48 Z_50 Y_52 X_55 Y_57 Z_60 X_63 Z_67 Y_70 X_72',
    '-i X_1 Z_2 Y_4 X_6 Y_8 Z_10 X_12 Z_15 X_17 Y_18 Z_20 X_23 Y_25 X_27 Z_30 Y_31 Z_34 X_36 Y_39 X_41 Z_45 Y_47 X_50 Z_52 Y_53 X_55 Z_57 Y_60 X_62 Z_65',
    '- Z_0 X_2 Y_3 X_5 Z_8 Y_12 Z_14 X_17 Y_19 Z_22 X_24 Y_27 Z_30 X_34 Y_36 Z_39 Y_42 X_45 Z_48 Y_51 X_53 Z_56 Y_59 X_62 Z_66 Y_70',
    '+i X_0 Z_1 Y_3 Z_4 X_6 Y_8 Z_10 X_12 Y_14 X_17 Z_19 Y_22 X_25 Z_26 Y_29 X_31 Z_33 Y_36 X_38 Z_41 Y_44 X_47 Z_49 Y_53 Z_57 Y_60',
    '-i X_0 Y_2 Z_3 X_4 Y_6 Z_7 X_9 Y_11 Z_12 X_14 Y_16 Z_18 X_20 Y_23 Z_25 X_27 Y_30 Z_32 X_34 Y_37 Z_39 X_41 Y_44 Z_47 X_50 Y_53',
    '+ X_0 Y_1 Z_3 X_4 Y_5 Z_7 X_9 Y_12 Z_15 X_18 Y_20 Z_22 X_24 Y_27 Z_29 X_31 Y_34 Z_36 X_38 Y_41 Z_44 X_47 Y_50 Z_53 X_56',
    '-i Y_0 X_2 Z_4 Y_5 X_7 Z_8 Y_10 X_13 Z_15 Y_18 X_20 Z_22 Y_24 X_26 Z_29 Y_32 X_34 Z_37 Y_39 X_42 Z_45 Y_48 X_50 Z_53',
    '- Z_1 X_3 Y_4 X_6 Z_8 Y_11 X_13 Z_16 Y_19 X_21 Z_23 Y_26 X_28 Z_30 Y_33 X_35 Z_38 Y_41 X_44 Z_46 Y_49 X_52 Z_55 Y_57',
    '+i X_0 Y_3 Z_4 X_5 Y_7 Z_9 X_12 Y_14 Z_16 X_18 Y_21 Z_24 X_27 Y_29 Z_31 X_34 Y_36 Z_38 X_41 Y_43 Z_45 X_48 Y_50 Z_53'
]

In [50]:
LLM_answers_pc = [mcp.PauliTerm(text = LLM_answers[i]) for i in range(len(LLM_answers))]
accuracy = [LLM_answers_pc[i] == answers[i] for i in range(min(len(LLM_answers), len(answers)))]
print(f'N_max = {N_max}, question_volume  = {iterations}, accuracy = {sum(accuracy)/len(accuracy)}')

N_max = 100, question_volume  = 10, accuracy = 0.0


In [51]:
accuracy

[False, False, False, False, False, False, False, False, False, False]